# Capture BLE Advertisments and SDR Data Asynchronously

## Usage
A python kernel containing all modules to run this notebook is found in [ble-rff-env](../ble-rff-env/).
To collect data, simply execute the cells in order.

## Dependencies
This Notebook uses the following code:

[capture_ble.py](../src/capture_ble.py) to scan for BLE advertisments asynchronously and write the detected devices to ```config['ble_log_path']```.

[capture_sdr.py](../src/capture_sdr.py) to import capture data from the PlutoSDR to ```config['sdr_data_path']```

In [ ]:
# Check Which Python Installation is Used.
# This cell should print the absolute path to ../ble-rff-env/bin/python3.exe
!which python3

In [ ]:
# Includes
import asyncio
import sys, os

# Add the src folder to Python path
sys.path.append(os.path.abspath("../src"))
from capture_sdr import PlutoSDRCapture
from capture_ble import BLECapture

In [ ]:
# Configuration
config = {
    'duration': 30,
    
    'sdr_center_freq': 2402e6,
    'sdr_sample_rate': 2000000,
    'sdr_num_samples': 2**20,
    'sdr_save_path': '../disk/raw',

    'ble_address_prefix': '06:05:04:03:02:01',
    'ble_rssi_threshold': -70,
    'ble_log_path': '../data/ble_log.csv'
}
# SDR and BLE Objects
ble = BLECapture(
    log_path=config['ble_log_path'],
    rssi_threshold=config['ble_rssi_threshold'],
    address_prefix=config['ble_address_prefix']
)

sdr = PlutoSDRCapture(
    center_freq=config['sdr_center_freq'],
    sample_rate=config['sdr_sample_rate'],
    num_samples=config['sdr_num_samples'],
    duration=config['duration'],
    save_path=config['sdr_save_path']
)

In [ ]:
# Asynchronous main method
async def main():
    async with asyncio.TaskGroup() as tg:
        ble_task = tg.create_task(ble.capture())
        sdr_task = tg.create_task(sdr.capture())

        await sdr_task
        ble.stop_capture()
        await ble_task

In [ ]:
# Run the main method to capture data
await main()